### Group 24

Shiref Khaled Elhalawany -  221100944

Ahmed Anis Hassan - 221100101 

Karim Ashraf Elsayed - 221100391

Kareem Shaheen - 221101524

# Part 2: Content-Based Recommendation

## 3. Feature Extraction and Vector Space Model

In this section, we implement **Text Feature Extraction** using a manually constructed **TF-IDF (Term Frequency-Inverse Document Frequency)** vectorizer. 

We will not use high-level libraries like `sklearn`'s `TfidfVectorizer` for the core logic. instead, we will build the process step-by-step:
1.  **Text Preprocessing**: Tokenization and stop-word removal.
2.  **Vocabulary Building**: Identifying unique terms and their document frequencies.
3.  **IDF Computation**: Calculating the inverse document frequency weights.
4.  **Vector Construction**: Transforming text data into a sparse TF-IDF matrix.


In [ ]:
import numpy as np
import pandas as pd
import re
import math
from collections import Counter
from scipy.sparse import csr_matrix, hstack, save_npz, vstack, diags
import os

RESULTS_DIR = "../results"
if not os.path.exists(RESULTS_DIR):
    os.makedirs(RESULTS_DIR)

### Subtask 1: Define the text source for items

We will extract the relevant text data from our items. We will combine `title` and `categories` into a single text string for each item.

In [ ]:
def get_item_text_data(df_items):
    print("Extracting item text data...")
    
    df_items['title'] = df_items['title'].fillna('')
    df_items['categories'] = df_items['categories'].fillna('')
    
    def clean_cat(c):
        if isinstance(c, str) and c.startswith("[") and c.endswith("]"):
             return c.replace("'", "").replace("[", "").replace("]", "").replace(",", " ")
        return str(c)

    df_items['text_source'] = df_items['title'] + " " + df_items['categories'].apply(clean_cat)
    
    print(f"Created text corpus for {len(df_items)} items.")
    return df_items['text_source'].tolist(), df_items['item_id'].tolist()

### Subtask 2: Basic text preprocessing

We perform tokenization (splitting text into words) and remove common stop-words.

In [ ]:
def manual_tokenize_and_clean(text_corpus):
    print("Preprocessing text (Tokenization & Stop-word removal)...")
    
    STOP_WORDS = set([
        'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', 'your', 'yours',
        'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', 'her', 'hers',
        'herself', 'it', 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves',
        'what', 'which', 'who', 'whom', 'this', 'that', 'these', 'those', 'am', 'is', 'are',
        'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does',
        'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until',
        'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into',
        'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down',
        'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here',
        'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more',
        'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so',
        'than', 'too', 'very', 's', 't', 'can', 'will', 'just', 'don', 'should', 'now', '&'
    ])
    
    processed_corpus = []
    
    for text in text_corpus:
        text = text.lower()
        tokens = re.findall(r'\b[a-z]{2,}\b', text)
        clean_tokens = [t for t in tokens if t not in STOP_WORDS]
        processed_corpus.append(clean_tokens)
        
    print(f"Preprocessed {len(processed_corpus)} documents.")
    return processed_corpus

### Subtask 3 & 4: Configure and Fit TF-IDF Vectorizer

We will build the vocabulary from our processed corpus, filtering rare words to keep dimensionality manageable. Then, we calculate the IDF for each word.

$$ IDF(t) = \log \left( \frac{N}{DF(t)} \right) $$
where $N$ is total documents and $DF(t)$ is document frequency of term $t$.

In [ ]:
def build_vocabulary_and_idf(processed_corpus, min_df=5):
    print(f"Building vocabulary (min_df={min_df})...")
    
    doc_freqs = Counter()
    N = len(processed_corpus)
    
    for tokens in processed_corpus:
        unique_tokens = set(tokens)
        for t in unique_tokens:
            doc_freqs[t] += 1
            
    total_unique_terms = len(doc_freqs)
    
    vocab = {}
    idf_scores = {}
    idx = 0
    
    sorted_terms = sorted(doc_freqs.keys()) 
    
    for term in sorted_terms:
        df = doc_freqs[term]
        if df >= min_df:
            vocab[term] = idx
            idf_scores[term] = math.log((N + 1) / (df + 1)) + 1
            idx += 1
            
    print(f"Validation: Processed {N} documents. Found {total_unique_terms} unique terms. "
          f"Kept {len(vocab)} terms with frequency >= {min_df}.")
    return vocab, idf_scores

### Subtask 4 (Continued): Transform to TF-IDF Matrix

We now calculate the TF-IDF vector for each document and store it in a sparse CSR matrix.
$$ TF\hbox{-}IDF(t, d) = TF(t, d) \times IDF(t) $$

In [ ]:
def compute_tf_idf_matrix(processed_corpus, vocab, idf_scores):
    print("Computing TF-IDF matrix...")
    rows = []
    cols = []
    data = []
    
    for doc_idx, tokens in enumerate(processed_corpus):
        term_counts = Counter(tokens)
        for term, count in term_counts.items():
            if term in vocab:
                col_idx = vocab[term]
                tf_idf_val = count * idf_scores[term]
                rows.append(doc_idx)
                cols.append(col_idx)
                data.append(tf_idf_val)

    matrix = csr_matrix((data, (rows, cols)), shape=(len(processed_corpus), len(vocab)))
    
    print("Performing L2 Normalization...")
    row_sums = np.array(matrix.power(2).sum(axis=1))
    row_norms = np.sqrt(row_sums)
    row_norms[row_norms == 0] = 1.0 
    row_norms = row_norms.flatten()
    
    inv_norms = 1.0 / row_norms
    from scipy.sparse import diags
    norm_matrix = diags(inv_norms) @ matrix
    
    print(f"TF-IDF matrix computation complete. Shape: {norm_matrix.shape}")
    first_row_norm = np.sqrt(np.sum(norm_matrix[0].data**2)) if norm_matrix.shape[0] > 0 else 0
    print(f"Validation: L2 norm of the first document vector: {first_row_norm:.4f}")
    
    return norm_matrix

### Subtask 5 & 6: Inspect and Validate

We check the sparsity of our matrix and save a summary of the vocabulary to the results.

In [ ]:
def validate_and_save_features(matrix, vocab, item_ids):
    print("\n--- Feature Space Validation ---")
    n_docs, n_terms = matrix.shape
    nnz = matrix.nnz
    sparsity = 1.0 - (nnz / (n_docs * n_terms))
    
    print(f"Matrix Shape: ({n_docs}, {n_terms})")
    print(f"Non-zero elements: {nnz})")
    print(f"Sparsity: {sparsity*100:.4f}%")
    
    vocab_list = sorted(vocab.items(), key=lambda x: x[1])
    df_vocab = pd.DataFrame(vocab_list, columns=['Term', 'Index'])
    
    full_vocab_path = os.path.join(RESULTS_DIR, "tfidf_vocabulary_full.csv")
    sample_vocab_path = os.path.join(RESULTS_DIR, "tfidf_vocabulary_sample.csv")
    
    df_vocab.to_csv(full_vocab_path, index=False)
    df_vocab.head(100).to_csv(sample_vocab_path, index=False)
    
    print(f"Saved full vocabulary to {full_vocab_path}")
    print(f"Saved top 100 vocabulary terms to {sample_vocab_path}")
    
    return df_vocab

## 3.2. Additional Features

We incorporate **numerical features** (`price`, `average_rating`) and **categorical features** (`categories`) to enrich the item representation.

### Subtask 1: Identify available additional features
We extract `price` and `average_rating` from the metadata. `price` often requires cleaning (removing '$', converting to float).

In [ ]:
def load_additional_features(df_items):
    print("\n--- Extracting Additional Features ---")
    
    df_features = df_items.copy()
    
    if 'price' not in df_features.columns:
        df_features['price'] = np.nan
    if 'average_rating' not in df_features.columns:
        df_features['average_rating'] = np.nan
        
    def clean_price(p):
        if isinstance(p, (int, float)):
            return float(p)
        if isinstance(p, str):
            match = re.search(r'(\d+\.?\d*)', p)
            if match:
                return float(match.group(1))
        return np.nan

    df_features['price_num'] = df_features['price'].apply(clean_price)
    
    def clean_rating(r):
        try:
            return float(r)
        except:
            return np.nan
            
    df_features['rating_num'] = df_features['average_rating'].apply(clean_rating)
    
    price_median = df_features['price_num'].median()
    rating_median = df_features['rating_num'].median()
    
    if pd.isna(price_median): price_median = 0.0
    if pd.isna(rating_median): rating_median = 3.0 
    
    df_features['price_num'] = df_features['price_num'].fillna(price_median)
    df_features['rating_num'] = df_features['rating_num'].fillna(rating_median)
    
    print(f"Extracted numerical features. Imputed Price Median: {price_median}, Rating Median: {rating_median}")
    return df_features[['item_id', 'price_num', 'rating_num', 'categories']]

### Subtask 2-4: Process features

**1. Numerical Features**: We normalize `price` and `rating` using **Min-Max Scaling** manually to bring them to [0, 1] range.

**2. Categorical Features**: We perform a simplified **One-Hot Encoding** for the top `K` most frequent categories.

In [ ]:
def process_numerical_features(df_features):
    print("Processing numerical features (Min-Max Scaling)...")
    
    p_min = df_features['price_num'].min()
    p_max = df_features['price_num'].max()
    if p_max > p_min:
        df_features['price_scaled'] = (df_features['price_num'] - p_min) / (p_max - p_min)
    else:
        df_features['price_scaled'] = 0.0
        
    r_min = df_features['rating_num'].min()
    r_max = df_features['rating_num'].max()
    if r_max > r_min:
        df_features['rating_scaled'] = (df_features['rating_num'] - r_min) / (r_max - r_min)
    else:
        df_features['rating_scaled'] = 0.5
        
    return df_features[['price_scaled', 'rating_scaled']].values

def process_categorical_features(df_features, top_k=20):
    print(f"Processing categorical features (Top {top_k} OHE)...")
    
    all_cats = []
    cat_series = df_features['categories'].astype(str)
    
    for entry in cat_series:
        clean = re.sub(r'[^a-zA-Z0-9\s]', ' ', entry)
        words = clean.split()
        all_cats.extend(words)
        
    cat_counts = Counter(all_cats)
    top_cats = [c[0] for c in cat_counts.most_common(top_k) if len(c[0]) > 2]
    top_cats = top_cats[:top_k]
    
    print(f"Top categories identified: {top_cats}")
    
    N = len(df_features)
    cat_matrix = np.zeros((N, len(top_cats)))
    
    for i, entry in enumerate(cat_series):
        entry_lower = entry.lower()
        for j, cat in enumerate(top_cats):
            if cat.lower() in entry_lower:
                cat_matrix[i, j] = 1.0
                
    return cat_matrix, top_cats

### Subtask 5: Combine features

We concatenate the sparse TF-IDF matrix with the dense numerical and categorical matrices.

In [ ]:
def combine_features(tfidf_matrix, num_features, cat_features):
    print("Combining all features...")
    
    num_sparse = csr_matrix(num_features)
    cat_sparse = csr_matrix(cat_features)
    
    final_matrix = hstack([tfidf_matrix, num_sparse, cat_sparse], format='csr')
    
    print(f"Final Feature Matrix Shape: {final_matrix.shape}")
    
    save_npz_path = os.path.join(RESULTS_DIR, "feature_matrix.npz")
    save_npz(save_npz_path, final_matrix)
    print(f"Feature matrix saved to {save_npz_path}")
    
    return final_matrix

## 3.3. Create item-feature matrix and document your feature selection

We have combined the features. Now we must **validate** the consistency of the feature space and **document** our rationale.

### Subtask 5 & 6: Validate and Check Scaling
We check if the different feature distincts (TF-IDF vs Numerical) have vastly different magnitudes. While TF-IDF is unit-length (L2), numerical features are [0, 1]. In high dimensions, unit vectors have small components. We verify statistics.

In [ ]:
def validate_item_feature_matrix(matrix):
    print("\n--- Validating Item-Feature Matrix ---")
    
    n_items, n_features = matrix.shape
    nnz = matrix.nnz
    sparsity = 1.0 - (nnz / (n_items * n_features))
    print(f"Final Matrix Shape: ({n_items}, {n_features})")
    print(f"Sparsity: {sparsity*100:.4f}%")
    
    sample_indices = np.random.choice(n_items, size=min(1000, n_items), replace=False)
    sample_matrix = matrix[sample_indices]
    
    max_val = sample_matrix.max()
    mean_val = sample_matrix.mean()
    
    print(f"Max value in sample: {max_val:.4f}")
    print(f"Mean value in sample: {mean_val:.6f} (Expected to be low due to sparsity)")
    
    row_sums = np.array(sample_matrix.power(2).sum(axis=1))
    row_norms = np.sqrt(row_sums).flatten()
    
    print(f"Average Row L2 Norm: {np.mean(row_norms):.4f} (TF-IDF base was 1.0)")
    
    if max_val > 10.0:
        print("WARNING: Some features have very high values. scaling might be off.")
    else:
        print("Scaling consistency check passed (Values roughly in expected range).")

### Subtask 7: Document feature selection rationale

We save a explanation of why we chose these features.

In [ ]:
def save_feature_selection_rationale():

    print("Saving feature selection rationale...")
    
    rationale = """
# Feature Selection Rationale for Book Recommendation

## 1. Text Features (TF-IDF)
- **Source**: 'title' combined with 'categories'.
- **Method**: TF-IDF (Term Frequency-Inverse Document Frequency).
- **Reason**: Books are content-rich items. Title and categories provide strong semantic signals about the book's topic/genre. TF-IDF downweights common words ensuring unique keywords drive similarity.

## 2. Numerical Features
- **Features**: 'price', 'average_rating'.
- **Method**: Min-Max Scaling [0, 1].
- **Reason**: 
    - **Price**: Users often prefer books in specific price ranges. Normalization ensures it doesn't dominate the vector space.
    - **Average Rating**: Acts as a proxy for quality. Higher rated items might be more 'summable' with other high quality items.

## 3. Categorical Features
- **Method**: One-Hot Encoding (Top 20 frequent categories).
- **Reason**: While 'categories' are in the text indices, explicit dimensions for major genres (Fiction, Mystery, etc.) allow the model to strictly cluster items of the same 'type' even if titles are very different.

## 4. Combination
- We stack these vectors. The final representation mixes semantic similarity (Text) with property similarity (Price/Quality/Genre).
"""
    
    path = os.path.join(RESULTS_DIR, "feature_selection_rationale.md")
    with open(path, "w") as f:
        f.write(rationale)
        
    print(f"Rationale saved to {path}")
    
    print("Feature Matrix Creation and Documentation Complete.")

# -------------------------------------------------------------------------
# 4. User Profile Construction
# -------------------------------------------------------------------------

We construct user profiles by computing the **weighted average** of the feature vectors of items they have rated. 

### Subtask 1-4: Build Profiles (Batched)
**Correction:** Due to large number of users, we process in batches and save intermediate sparse matrices to avoid memory errors.

For each user $u$:
$$ \vec{p}_u = \frac{\sum_{i \in I_u} r_{ui} \cdot \vec{f}_i}{\sum_{i \in I_u} r_{ui}} $$
where $\vec{f}_i$ is the item feature vector and $r_{ui}$ is the rating.

In [ ]:
def build_user_profiles(df_interactions, item_feature_matrix, df_items_map, batch_size=5000):
    print("\n--- User Profile Construction (Batched) ---")
    
    parts_dir = os.path.join(RESULTS_DIR, "user_profiles_parts")
    if not os.path.exists(parts_dir):
        os.makedirs(parts_dir)
        
    print("Mapping item IDs to feature matrix indices...")
    item_to_idx = {iid: idx for idx, iid in enumerate(df_items_map['item_id'])}
    
    user_groups = df_interactions.groupby('user_id')
    n_users = len(user_groups)
    print(f"Total users to process: {n_users}")
    
    current_batch_vectors = []
    current_batch_ids = []
    saved_batches = []
    
    count = 0
    n_features = item_feature_matrix.shape[1]
    
    for uid, group in user_groups:
        valid_indices = []
        ratings = []
        
        for _, row in group.iterrows():
            iid = row['item_id']
            r = row['rating']
            if iid in item_to_idx:
                valid_indices.append(item_to_idx[iid])
                ratings.append(r)
        
        if not valid_indices:
            user_vec_sparse = csr_matrix((1, n_features))
        else:
            item_vecs = item_feature_matrix[valid_indices]
            ratings_arr = np.array(ratings).reshape(-1, 1)
            weighted_sum_dense = item_vecs.multiply(ratings_arr).sum(axis=0)
            
            total_rating = np.sum(ratings)
            if total_rating > 0:
                weighted_sum_dense /= total_rating
                
            user_vec_sparse = csr_matrix(weighted_sum_dense)
            
        current_batch_vectors.append(user_vec_sparse)
        current_batch_ids.append(uid)
        count += 1
        
        if len(current_batch_vectors) >= batch_size:
            batch_matrix = vstack(current_batch_vectors)

            row_sums = np.array(batch_matrix.power(2).sum(axis=1))
            row_norms = np.sqrt(row_sums).flatten()
            row_norms[row_norms == 0] = 1.0
            inv_norms = 1.0 / row_norms
            batch_matrix = diags(inv_norms) @ batch_matrix
            
            batch_idx = len(saved_batches)
            filename = f"user_profiles_part_{batch_idx}.npz"
            path = os.path.join(parts_dir, filename)
            save_npz(path, batch_matrix)
            
            id_path = path.replace(".npz", "_ids.csv")
            pd.DataFrame(current_batch_ids, columns=['user_id']).to_csv(id_path, index=False)
            
            saved_batches.append(path)
            print(f"Saved batch {batch_idx}: {batch_matrix.shape} to {filename}")
            
            current_batch_vectors = []
            current_batch_ids = []
            import gc; gc.collect()
            
        if count % 10000 == 0:
            print(f"Processed {count}/{n_users} users...")

    if current_batch_vectors:
        batch_matrix = vstack(current_batch_vectors)
        
        row_sums = np.array(batch_matrix.power(2).sum(axis=1))
        row_norms = np.sqrt(row_sums).flatten()
        row_norms[row_norms == 0] = 1.0
        inv_norms = 1.0 / row_norms
        batch_matrix = diags(inv_norms) @ batch_matrix
        
        batch_idx = len(saved_batches)
        filename = f"user_profiles_part_{batch_idx}.npz"
        path = os.path.join(parts_dir, filename)
        save_npz(path, batch_matrix)
        
        id_path = path.replace(".npz", "_ids.csv")
        pd.DataFrame(current_batch_ids, columns=['user_id']).to_csv(id_path, index=False)
        saved_batches.append(path)
        print(f"Saved final batch {batch_idx}: {batch_matrix.shape} to {filename}")
        
    print(f"\nUser Profile Construction Complete. Saved {len(saved_batches)} parts.")
    return saved_batches

## 4.2. Handle cold-start users (Popular Item Features Strategy)

### Subtask 1: Define what a cold-start user is
A **Cold-Start User** is a new user who has not interacted with (rated or viewed) any items in the system yet.
Since there is no historical data to compute a personalized profile or find similar users, traditional Collaborative Filtering fails.
To address this, we use a **Popularity-Based** or **Demographic-Based** strategy to generate an initial profile.
Here, we use the **Popular Item Features** strategy: we assume a new user is likely to be interested in what the majority of people like.

In [ ]:
def identify_popular_items(df_interactions, top_n=50):
    print(f"Identifying top {top_n} popular items...")
    popular_counts = df_interactions['item_id'].value_counts().head(top_n)
    popular_item_ids = popular_counts.index.tolist()
    print(f"Found {len(popular_item_ids)} popular items.")
    return popular_item_ids

In [ ]:
def extract_popular_vectors(popular_item_ids, item_feature_matrix, df_items_map):
    print("Extracting feature vectors for popular items...")
    item_to_idx = {iid: idx for idx, iid in enumerate(df_items_map['item_id'])}
    
    indices = []
    for iid in popular_item_ids:
        if iid in item_to_idx:
            indices.append(item_to_idx[iid])
            
    if not indices:
        print("Warning: No popular items found in feature matrix.")
        return None
        
    pop_vectors = item_feature_matrix[indices]
    print(f"Extracted shape: {pop_vectors.shape}")
    return pop_vectors

In [ ]:
def construct_cold_start_profile(df_interactions, item_feature_matrix, df_items_map, top_n=50):
    print("\n--- Constructing Cold-Start User Profile ---")
    
    pop_ids = identify_popular_items(df_interactions, top_n)
    
    pop_vectors = extract_popular_vectors(pop_ids, item_feature_matrix, df_items_map)
    
    if pop_vectors is None:
        return csr_matrix((1, item_feature_matrix.shape[1]))
    

    cold_start_vec = pop_vectors.sum(axis=0) / pop_vectors.shape[0]
    cold_start_vec = csr_matrix(cold_start_vec)
    
    norm = np.linalg.norm(cold_start_vec.data)
    if norm > 0:
        cold_start_vec = cold_start_vec / norm
        
    print(f"Cold-start profile constructed. Shape: {cold_start_vec.shape}")
    return cold_start_vec

### Subtask 6: Explain when this profile is used
This **Cold-Start Profile** is used whenever the system encounters a user with **zero interactions** (or fewer than a threshold, e.g., < 3).
Instead of returning random items, we use this profile to calculate cosine similarity against all items, effectively returning items that differ slightly from pure popularity but are semantically similar to the popular 'consensus'.

### Subtask 7: Justify the strategy
**Justification**:
1.  **Robustness**: Popular items are statistically significant 'safe bets' for unknown users.
2.  **Content-Aware**: By averaging *features* of popular items rather than just recommending IDs, we can recommend *niche* items that are similar content-wise to popular ones, improving diversity (Serendipity) compared to a simple "Top-N Popular" list.
3.  **Simplicity**: It effectively boosts the user into the vector space immediately without requiring expensive model retraining (SVD) or demographic data lookup.

In [ ]:
def run_cold_start_module(df_interactions, item_feature_matrix, df_items_map):
    print("\n=== Running Cold-Start Module ===")
    cold_user_profile = construct_cold_start_profile(df_interactions, item_feature_matrix, df_items_map)
    
    save_path = os.path.join(RESULTS_DIR, "cold_start_profile.npz")
    save_npz(save_path, cold_user_profile)
    print(f"Saved cold-start profile to {save_path}")
    return cold_user_profile

## 5. Similarity Computation and Recommendation

We now calculate the similarity between the user profile and all items to generate recommendations.

### Subtask 1: Ensure vector space alignment
We verify that the User Vector and Item Feature Matrix share the same number of dimensions before proceeding.

In [ ]:
def check_vector_alignment(user_vec, item_matrix):
    print("\n--- Verifying Vector Alignment ---")
    user_dim = user_vec.shape[1]
    item_dim = item_matrix.shape[1]
    
    if user_dim != item_dim:
        raise ValueError(f"Dimension mismatch! User: {user_dim}, Item: {item_dim}")
        
    print(f"Alignment Verified. Dimensions: {user_dim}")
    return True

### Subtask 2: Define cosine similarity formally

**Cosine Similarity** measures the cosine of the angle between two non-zero vectors. 
$$ \text{similarity} = \cos(\theta) = \frac{\mathbf{A} \cdot \mathbf{B}}{||\mathbf{A}|| \cdot ||\mathbf{B}||} $$

Since our vectors (both TF-IDF/Item vectors and User vectors) effectively undergo L2 normalization during their construction, their magnitudes are close to 1. Thus, the calculation simplifies to the **Dot Product**:
$$ \text{similarity} \approx \mathbf{A} \cdot \mathbf{B} $$

In [ ]:
def compute_cosine_similarity(user_vec, item_matrix):
    print("Computing Cosine Similarity...")
    
    similarity_scores = item_matrix.dot(user_vec.T)
    
    similarity_scores = similarity_scores.toarray().flatten()
    
    print(f"Computed {len(similarity_scores)} similarity scores.")
    return similarity_scores

In [ ]:
def save_similarity_scores(scores, item_ids, filename="similarity_scores.csv"):
    print(f"Saving similarity scores to {filename}...")
    path = os.path.join(RESULTS_DIR, filename)
    
    df_scores = pd.DataFrame({
        'item_id': item_ids,
        'score': scores
    })
    
    df_scores = df_scores.sort_values(by='score', ascending=False)
    
    df_scores.to_csv(path, index=False)
    print("File saved.")
    return df_scores

In [ ]:
def verify_similarity_results(scores):
    print("Verifying similarity scores...")
    min_s = scores.min()
    max_s = scores.max()
    
    print(f"Range: [{min_s:.4f}, {max_s:.4f}]")
    
    if min_s < -1.01 or max_s > 1.01:
        print("WARNING: Scores out of expected cosine range [-1, 1]. Check normalization.")
    else:
        print("Verification Passed: Scores within valid range.")

### Subtask 6: Explain what similarity scores mean

The **Similarity Score** (ranging from -1 to 1) quantifies how close the item's content is to the user's preference profile.
- **Approaching 1**: The item is very semantically similar to what the user likes (or the popular consensus in the cold-start case).
- **Approaching 0**: The item is orthogonal (unrelated) to the user's profile.
- **Approaching -1**: The item is opposite (rare in positive-only Feature spaces like TF-IDF, but possible mathematically).

In [ ]:
def run_similarity_module(user_vec, item_matrix, df_items_map):
    print("\n=== Running Similarity Component ===")
    
    check_vector_alignment(user_vec, item_matrix)
    
    scores = compute_cosine_similarity(user_vec, item_matrix)
    
    verify_similarity_results(scores)
    
    df_scored = save_similarity_scores(scores, df_items_map['item_id'], filename="cold_start_similarity_scores.csv")
    
    return df_scored

## 5.2. Generate Top-N Recommendations

We rank items by their similarity scores and exclude items the user has already rated to generate the final Top-N list.

In [ ]:

def get_target_users(df_interactions, cold_start_profile, n_existing=2):
    print("\n--- Selecting Target Users ---")
    target_users = {}
    
    target_users['Cold_Start_User'] = {
        'type': 'cold',
        'data': cold_start_profile
    }
    
    user_counts = df_interactions['user_id'].value_counts()
    active_users = user_counts.head(50).index.tolist()
    selected_existing = active_users[:n_existing]
    
    for uid in selected_existing:
        target_users[f'User_{uid}'] = {
            'type': 'existing',
            'data': uid 
        }
        
    print(f"Selected {len(target_users)} target users: {list(target_users.keys())}")

    return target_users

In [ ]:
def get_target_users_1(df_interactions, cold_start_profile, n_existing=2, user_col='user_id'):
    print("\n--- Selecting Target Users ---")
    target_users = {
        'Cold_Start_User': {'type': 'cold', 'data': cold_start_profile}
    }

    if df_interactions is None or len(df_interactions) == 0:
        print("WARNING: df_interactions is empty. No existing users can be selected.")
        return target_users

    if user_col not in df_interactions.columns:
        raise ValueError(f"'{user_col}' not found in df_interactions columns: {list(df_interactions.columns)}")

    user_counts = df_interactions[user_col].dropna().value_counts()
    if len(user_counts) == 0:
        print(f"WARNING: No valid users found in column '{user_col}'.")
        return target_users

    active_users = user_counts.head(50).index.tolist()
    selected_existing = active_users[:n_existing]

    for uid in selected_existing:
        target_users[f'User_{uid}'] = {'type': 'existing', 'data': uid}

    print(f"Selected {len(target_users)} target users: {list(target_users.keys())}")
    return target_users

In [ ]:
def get_user_rated_items(df_interactions, user_id):
    if user_id is None: 
        return set()
        
    user_data = df_interactions[df_interactions['user_id'] == user_id]
    rated_items = set(user_data['item_id'].unique())
    return rated_items

In [ ]:
def rank_and_filter_items(similarity_scores, item_ids, rated_items_set):
    print(f"Ranking items... (Total candidates: {len(item_ids)}) ")
    
    candidates = []
    
    for score, iid in zip(similarity_scores, item_ids):
        if iid not in rated_items_set:
            candidates.append((score, iid))
            
    print(f"Filtered out {len(item_ids) - len(candidates)} rated items. Remaining candidates: {len(candidates)}")
    
    candidates.sort(key=lambda x: x[0], reverse=True)
    
    return candidates

In [ ]:
def generate_top_n_recommendations(sorted_candidates, top_n_list=[10, 20]):
    results = {}
    for n in top_n_list:
        results[n] = sorted_candidates[:n]
        
    return results

In [ ]:
def save_recommendations(user_label, top_n_results, df_items_map):
    print(f"Saving recommendations for {user_label}...")
    
    item_map = df_items_map.set_index('item_id').to_dict('index')
    
    max_n = max(top_n_results.keys())
    top_items = top_n_results[max_n]
    
    data = []
    rank = 1
    for score, iid in top_items:
        info = item_map.get(iid, {})
        row = {
            'Rank': rank,
            'User': user_label,
            'Item_ID': iid,
            'Score': round(score, 6),
            'Title': info.get('title', 'Unknown'),
            'Category': info.get('categories', 'Unknown')
        }
        data.append(row)
        rank += 1
        
    df_recs = pd.DataFrame(data)
    filename = f"recommendations_{user_label}.csv"
    path = os.path.join(RESULTS_DIR, filename)
    df_recs.to_csv(path, index=False)
    
    print(f"Saved to {path}")
    return df_recs

### Subtask 8: Explanation of ranking logic

The ranking is purely based on the **Cosine Similarity Score**.
1.  We calculate the similarity between the User Profile (weighted average of their history) and every Item Vector.
2.  We **exclude** items the user has already rated to ensure novelty.
3.  We **sort** the remaining items in descending order of similarity.
4.  The top items represented the "best match" in the vector space.

In [ ]:
def run_recommendation_pipeline(df_interactions, item_feature_matrix, df_items_map, cold_start_profile):
    print("\n=== Running Recommendation Pipeline ===")
    
    target_users = get_target_users(df_interactions, cold_start_profile, n_existing=2)
    
    all_recs = []
    
    for label, info in target_users.items():
        print(f"\nProcessing {label}...")
        
        if info['type'] == 'cold':
            user_vec = info['data']
            rated_items = set()
        else:
            uid = info['data']
            rated_items = get_user_rated_items(df_interactions, uid)
            
            item_to_idx = {iid: idx for idx, iid in enumerate(df_items_map['item_id'])}
            indices = [item_to_idx[iid] for iid in rated_items if iid in item_to_idx]
            
            if not indices:
                 user_vec = csr_matrix((1, item_feature_matrix.shape[1]))
            else:
                user_data = df_interactions[df_interactions['user_id'] == uid]
                ratings_map = temp_r_map = dict(zip(user_data['item_id'], user_data['rating']))
                
                valid_indices = []
                valid_ratings = []
                for iid in rated_items:
                    if iid in item_to_idx:
                        valid_indices.append(item_to_idx[iid])
                        valid_ratings.append(ratings_map[iid])
                        
                item_vecs = item_feature_matrix[valid_indices]
                ratings_arr = np.array(valid_ratings).reshape(-1, 1)
                weighted = item_vecs.multiply(ratings_arr).sum(axis=0)
                weighted /= np.sum(valid_ratings)
                user_vec = csr_matrix(weighted)
                
                norm = np.linalg.norm(user_vec.data)
                if norm > 0: user_vec = user_vec / norm
        
        scores = compute_cosine_similarity(user_vec, item_feature_matrix)
        
        item_ids = df_items_map['item_id'].tolist()
        candidates = rank_and_filter_items(scores, item_ids, rated_items)
        
        top_results = generate_top_n_recommendations(candidates, [10, 20])
        
        df_rec = save_recommendations(label, top_results, df_items_map)
        all_recs.append(df_rec)
        
        print(f"Top 5 for {label}: ")
        print(df_rec[['Title', 'Score']].head(5))
    
    return all_recs

# 6. k-Nearest Neighbors (k-NN)

## 6.1. Implement Item-Based k-NN

We implement a memory-efficient Item-based k-NN. Instead of computing the full $N \times N$ similarity matrix (which can be huge), we compute similarities row-by-row and only store the **Top-K** nearest neighbors for each item.

### Subtask 1: Choose the item representation
We use the **Feature Matrix** constructed in Section 3 (`final_feature_matrix`) as the item representation. It combines TF-IDF, numerical, and categorical features.

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix, vstack
import heapq

def compute_top_k_similar_items(item_feature_matrix, df_items_map, k_list=[10, 20]):
    print("\n--- Computing Item-Item Similarity (Top-K) ---")
    
    max_k = max(k_list)
    n_items = item_feature_matrix.shape[0]
    item_ids = df_items_map['item_id'].tolist()
    
    item_neighbors = {}

    batch_size = 100
    
    print(f"Processing {n_items} items in batches of {batch_size}...")
    
    for start_idx in range(0, n_items, batch_size):
        end_idx = min(start_idx + batch_size, n_items)
        
        batch_vecs = item_feature_matrix[start_idx:end_idx]
        
        sim_batch = batch_vecs.dot(item_feature_matrix.T)
        
        if isinstance(sim_batch, csr_matrix):
            sim_batch = sim_batch.toarray()
            
        for i in range(len(sim_batch)):
            current_item_idx = start_idx + i
            current_item_id = item_ids[current_item_idx]
            
            scores = sim_batch[i]
            
            scores[current_item_idx] = -1.0
            
            top_indices = np.argsort(scores)[-max_k:][::-1]
            
            neighbors = []
            for idx in top_indices:
                score = scores[idx]
                if score > 0:
                    neighbor_id = item_ids[idx]
                    neighbors.append((float(score), neighbor_id))
            
            item_neighbors[current_item_id] = neighbors
        
        if (start_idx // batch_size) % 10 == 0:
             print(f"Processed {end_idx}/{n_items} items...")
             
    print("Top-K Neighbor computation complete.")
    return item_neighbors

In [ ]:
def predict_rating_knn(user_id, target_item_id, item_neighbors, user_ratings_map, k=20):

    neighbors = item_neighbors.get(target_item_id, [])
    
    top_k_neighbors = neighbors[:k]
    
    weighted_sum = 0.0
    sum_sim = 0.0
    
    count_contributors = 0
    
    for score, neighbor_id in top_k_neighbors:
        if neighbor_id in user_ratings_map:
            r_uj = user_ratings_map[neighbor_id]
            weighted_sum += score * r_uj
            sum_sim += abs(score)
            count_contributors += 1
            
    if count_contributors == 0 or sum_sim == 0:
        return None 
        
    prediction = weighted_sum / sum_sim
    return prediction

In [ ]:
def generate_knn_recommendations(target_users, item_neighbors, df_interactions, df_items_map, k_list=[10, 20]):
    print("\n--- Generating KNN Recommendations ---")
    all_results = []
    
    all_item_ids = df_items_map['item_id'].tolist()
    item_map = df_items_map.set_index('item_id').to_dict('index')
    
    for user_label, info in target_users.items():
        if info['type'] == 'cold':
            print(f"Skipping {user_label} for KNN (Requires history).")
            continue
            
        user_id = info['data']
        print(f"generating for {user_label} ({user_id})...")
        
        user_data = df_interactions[df_interactions['user_id'] == user_id]
        user_ratings_map = dict(zip(user_data['item_id'], user_data['rating']))
        rated_items = set(user_ratings_map.keys())
        
        candidate_set = set()
        for rated_item in rated_items:
            neighbors = item_neighbors.get(rated_item, [])
            for s, n_id in neighbors:
                if n_id not in rated_items:
                    candidate_set.add(n_id)
        
        print(f"Identified {len(candidate_set)} candidate items via neighbor expansion.")
        
        predictions = []
        for k in k_list:
            pass
            
        max_k = max(k_list)
        
        candidates_scored = []
        for item_id in candidate_set:
            pred = predict_rating_knn(user_id, item_id, item_neighbors, user_ratings_map, k=max_k)
            if pred is not None:
                candidates_scored.append((pred, item_id))
                
        candidates_scored.sort(key=lambda x: x[0], reverse=True)
        
        top_20 = candidates_scored[:20]
        
        out_data = []
        rank = 1
        for score, iid in top_20:
            meta = item_map.get(iid, {})
            out_data.append({
                'Rank': rank,
                'User': user_id,
                'Item_ID': iid,
                'Predicted_Rating': round(score, 4),
                'Title': meta.get('title', 'Unknown'),
                'Method': f'Item-KNN (k={max_k})'
            })
            rank += 1
            
        df_out = pd.DataFrame(out_data)
        filename = f"knn_recommendations_{user_id}.csv"
        path = os.path.join(RESULTS_DIR, filename)
        df_out.to_csv(path, index=False)
        print(f"Saved KNN recs to {path}")
        all_results.append(df_out)
        
    return all_results

## 6.2. Compare Content-Based and k-NN Approaches

We compare the two methods using a consistent evaluation setup.

In [ ]:
def evaluate_and_compare(target_users, df_interactions, item_feature_matrix, item_neighbors, df_items_map):
    print("\n=== Comparing Content-Based vs k-NN (Leave-One-Out Evaluation) ===")
    
    results = []
    
    for label, info in target_users.items():
        if info['type'] != 'existing':
            continue
            
        user_id = info['data']
        
        user_data = df_interactions[df_interactions['user_id'] == user_id]
        if len(user_data) < 2:
            continue
            
        hidden_item = user_data.iloc[-1]['item_id']
        train_items = set(user_data.iloc[:-1]['item_id'].unique())
        train_map = dict(zip(user_data.iloc[:-1]['item_id'], user_data.iloc[:-1]['rating']))
        
        print(f"Evaluating User {user_id}. Hidden Item: {hidden_item}")
        
        item_to_idx = {iid: idx for idx, iid in enumerate(df_items_map['item_id'])}
        indices = [item_to_idx[iid] for iid in train_items if iid in item_to_idx]
        
        cb_hit = 0
        knn_hit = 0
        
        if indices:
            item_vecs = item_feature_matrix[indices]
            ratings_arr = np.array([train_map[iid] for iid in train_items if iid in item_to_idx]).reshape(-1, 1)
            user_vec = item_vecs.multiply(ratings_arr).sum(axis=0)
            user_vec = csr_matrix(user_vec / np.sum(ratings_arr))
            if np.linalg.norm(user_vec.data) > 0:
                user_vec = user_vec / np.linalg.norm(user_vec.data)
                
            scores = compute_cosine_similarity(user_vec, item_feature_matrix)
            
            cb_candidates = []
            all_ids = df_items_map['item_id'].tolist()
            for s, iid in zip(scores, all_ids):
                if iid not in train_items:
                    cb_candidates.append((s, iid))
            cb_candidates.sort(key=lambda x: x[0], reverse=True)
            
            top_10 = [c[1] for c in cb_candidates[:10]]
            if hidden_item in top_10:
                cb_hit = 1
        
        negatives = []
        import random
        while len(negatives) < 100:
            i_rand = random.choice(all_ids)
            if i_rand not in train_items and i_rand != hidden_item:
                negatives.append(i_rand)
                
        test_candidates = [hidden_item] + negatives
        
        knn_scores = []
        for cand in test_candidates:
            score = predict_rating_knn(user_id, cand, item_neighbors, train_map, k=20)
            if score is None: score = 0 
            knn_scores.append((score, cand))
            
        knn_scores.sort(key=lambda x: x[0], reverse=True)
        top_10_knn = [c[1] for c in knn_scores[:10]]
        
        if hidden_item in top_10_knn:
            knn_hit = 1
            
        results.append({
            'User': user_id,
            'CB_Hit_10': cb_hit,
            'KNN_Hit_10': knn_hit,
            'Hidden_Item': hidden_item
        })
        
    df_res = pd.DataFrame(results)
    print("\n--- Evaluation Results (Hit Rate @ 10) ---")
    print(df_res)
    save_path = os.path.join(RESULTS_DIR, "method_comparison.csv")
    df_res.to_csv(save_path, index=False)

    
    cb_acc = df_res['CB_Hit_10'].mean()
    knn_acc = df_res['KNN_Hit_10'].mean()
    
    print("\n--- Interpretation ---")
    print(f"Content-Based Hit Rate: {cb_acc:.2f}")
    print(f"Item-KNN Hit Rate: {knn_acc:.2f}")
    
    if knn_acc > cb_acc:
        print("Conclusion: k-NN performed better. Collaborative signals (ratings) might be stronger than content Metadata here.")
    elif cb_acc > knn_acc:
        print("Conclusion: Content-Based performed better. Metadata specificities might be more effective than sparse rating overlaps.")
    else:
        print("Conclusion: Both methods performed similarly.")
        
    return df_res

# 7. Complete Numerical Example

## 7.1. Step-by-Step Numerical Walkthrough
This section provides a detailed numerical example using a small subset (3-5 items) to transparently demonstrate the calculations behind the scenes.

### Subtask 1: Select 3–5 sample items

In [ ]:
def select_sample_items(df_items, n=5):
    print(f"\n--- [Numerical Example] Selecting {n} Sample Items ---")
    
    sample = df_items.head(n).copy()
    if 'text_source' not in sample.columns:
        sample['text_source'] = sample['title'].fillna('') + " " + sample['categories'].fillna('')
        
    for i, row in sample.iterrows():
        print(f"Item {row['item_id']}: {row['title']} | Text: {row['text_source'][:50]}...")
        
    return sample

In [ ]:
def run_step_by_step_tfidf(sample_df):
    print("\n--- [Numerical Example] TF-IDF Calculation ---")
    
    texts = sample_df['text_source'].tolist()
    processed_docs = manual_tokenize_and_clean(texts)
    
    unique_terms = sorted(set(term for doc in processed_docs for term in doc))
    vocab = {term: i for i, term in enumerate(unique_terms)}
    print(f"\nSample Vocabulary ({len(vocab)} terms): {list(vocab.keys())}")
    
    tfs = []
    print("\nTerm Frequencies (TF):")
    for i, doc in enumerate(processed_docs):
        counts = Counter(doc)
        row = [counts[term] for term in unique_terms]
        tfs.append(row)
        print(f"Doc {i} ({sample_df.iloc[i]['item_id']}): {dict(zip(unique_terms, row))}")
    
    N = len(processed_docs)
    dfs = [sum(1 for doc in processed_docs for term in unique_terms if term in doc) for term in unique_terms]

    dfs = []
    for term in unique_terms:
        count = sum(1 for doc in processed_docs if term in doc)
        dfs.append(count)
    
    idfs = []
    print(f"\nInverse Document Frequencies (IDF) [log((N+1)/(df+1)) + 1]:")
    for term, df in zip(unique_terms, dfs):
        val = math.log((N + 1) / (df + 1)) + 1
        idfs.append(val)
        print(f"Term '{term}': DF={df}, IDF={val:.4f}")
        
    print("\nTF-IDF Matrix (TF * IDF) before Normalization:")
    tfidf_matrix = []
    for i, tf_row in enumerate(tfs):
        vec = [tf * idf for tf, idf in zip(tf_row, idfs)]
        tfidf_matrix.append(vec)
        vec_str = ", ".join([f"{v:.2f}" for v in vec])
        print(f"Doc {i}: [{vec_str}]")
        
    print("\nL2 Normalization:")
    final_matrix = []
    for i, vec in enumerate(tfidf_matrix):
        norm = math.sqrt(sum(v*v for v in vec))
        if norm > 0:
            norm_vec = [v/norm for v in vec]
        else:
            norm_vec = vec
        final_matrix.append(norm_vec)
        norm_vec_str = ", ".join([f"{v:.3f}" for v in norm_vec])
        print(f"Doc {i} Normalized: [{norm_vec_str}]")
        
    df_tfidf = pd.DataFrame(final_matrix, columns=unique_terms, index=sample_df['item_id'])
    
    df_tfidf.to_csv(os.path.join(RESULTS_DIR, "numerical_example_tfidf.csv"))
    return df_tfidf

In [ ]:
def define_sample_user(sample_df):
    print("\n--- [Numerical Example] Defining Sample User ---")
    items = sample_df['item_id'].tolist()
    if len(items) < 2:
         print("Not enough items to define sample user.")
         return {}
         
    user_ratings = {
        items[0]: 5.0, 
        items[1]: 2.0 
    }
    if len(items) > 2:
         user_ratings[items[2]] = 4.0
    
    print(f"User Ratings: {user_ratings}")
    return user_ratings

In [ ]:
def construct_sample_profile(user_ratings, df_tfidf_sample):
    print("\n--- [Numerical Example] User Profile Construction ---")
    
    n_features = df_tfidf_sample.shape[1]
    terms = df_tfidf_sample.columns.tolist()
    
    weighted_sum = np.zeros(n_features)
    total_rating = 0.0
    
    print("Calculation (Sum[r * v] / Sum[r]):")
    
    for iid, rating in user_ratings.items():
        if iid in df_tfidf_sample.index:
            vec = df_tfidf_sample.loc[iid].values
            weighted_sum += rating * vec
            total_rating += rating
            print(f" + (Rating {rating}) * Vector[{iid}]")
            
    if total_rating > 0:
        user_profile = weighted_sum / total_rating
    else:
        user_profile = weighted_sum
        
    norm = np.linalg.norm(user_profile)
    if norm > 0:
        user_profile = user_profile / norm
        
    print(f"\nFinal Normalized User Profile Vector (Top 5 features):")
    top_indices = np.argsort(user_profile)[::-1][:5]
    for idx in top_indices:
        print(f"'{terms[idx]}': {user_profile[idx]:.4f}")
        
    return user_profile

In [ ]:
def compute_sample_similarity_and_rank(user_profile, df_tfidf_sample):
    print("\n--- [Numerical Example] Similarity & Ranking ---")
    
    results = []
    
    for iid, row in df_tfidf_sample.iterrows():
        item_vec = row.values
        score = np.dot(user_profile, item_vec)
        results.append((score, iid))
        print(f"Item {iid}: Score = Dot(User, Item) = {score:.6f}")
        
    results.sort(key=lambda x: x[0], reverse=True)
    
    print("\n--- Top Recommendations --- ")
    rank = 1
    out_list = []
    for score, iid in results:
        print(f"{rank}. Item {iid} (Score: {score:.4f})")
        out_list.append({'Rank': rank, 'Item_ID': iid, 'Score': score})
        rank += 1
        
    pd.DataFrame(out_list).to_csv(os.path.join(RESULTS_DIR, "numerical_example_results.csv"), index=False)
    return results

In [ ]:
def run_numerical_example_pipeline(df_items):
    print("\n==============================================")
    print("       STARTING SECTION 7: NUMERICAL EXAMPLE       ")
    print("==============================================")
    
    sample = select_sample_items(df_items, n=5)
    
    df_tfidf_sample = run_step_by_step_tfidf(sample)
    
    user_ratings = define_sample_user(sample)
    
    user_profile = construct_sample_profile(user_ratings, df_tfidf_sample)
    
    compute_sample_similarity_and_rank(user_profile, df_tfidf_sample)
    
    print("\n[Numerical Example Completed Successfully]")